# Q1: Function Logging Decorator

Create a decorator:
```
@log_function
```
that logs when a function starts and finishes.

The decorator should work with functions that receive different kinds of arguments.

Test Functions
```
def add(a, b):
    pass

def greet(name):
    pass

def calculate_average(numbers):
    pass
```
Test Input
```
add(10, 20)

greet("Arsh")

calculate_average([85, 78, 92, 88])
```

The decorator should work without modifying the original function logic.

---

In [3]:
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"[START] Calling '{func.__name__}' with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[END]   '{func.__name__}' returned: {result}")
        return result
    return wrapper


@logger
def add(a,b):
    return a + b

@logger
def greet(name, greeting = "Hello"):
    return f"{greeting}, {name}!"

@logger
def calculate_average(numbers):
    if not numbers:
        return 0
    return sum(numbers) / len(numbers)

add(10,20)
print()
greet("Arsh")
print()
calculate_average([85, 78, 92, 88])
print()



[START] Calling 'add' with args=(10, 20), kwargs={}
[END]   'add' returned: 30

[START] Calling 'greet' with args=('Arsh',), kwargs={}
[END]   'greet' returned: Hello, Arsh!

[START] Calling 'calculate_average' with args=([85, 78, 92, 88],), kwargs={}
[END]   'calculate_average' returned: 85.75



---

# Q2: Timer Decorator & Decorator Chaining

Create a decorator:
```
@timer
```
that measures how long a function takes to execute.

Then create:
```python
@log_function
@timer
def process_data():
    ...
```

Reverse the order:
```python
@timer
@log_function
def process_data():
    ...
```
and compare the execution order.

Finally use:
```python
from functools import wraps
```
to preserve the wrapped function's metadata.

---

### Test Functions

Use functions that perform different amounts of work.

Test 1
```python
def calculate_sum():
    total = 0

    for i in range(100000):
        total += i

    return total
```
Test 2
```python
def calculate_large_sum():
    total = 0

    for i in range(1000000):
        total += i

    return total
```
Test 3
```python
def process_list():
    numbers = list(range(100000))
    return sorted(numbers, reverse=True)
```

Apply the decorators and compare the behavior.

---

In [20]:
from functools import wraps
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Function {func.__name__} took {end_time - start_time:.6f} seconds to execute.")
        return result
    return wrapper

def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] Calling '{func.__name__}' with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[END]   '{func.__name__}' returned: {result}")
        return result
    return wrapper

print("\n" + "=" * 60)
print("ORDER 1: @logger then @timer")
print("=" * 60)

@logger
@timer
def calculate_sum(): 
    total = 0 
    for i in range(100000): 
        total += i 
    
    return total

_ = calculate_sum()


print("\n" + "=" * 60)
print("ORDER 2: @timer then @logger")
print("=" * 60)

@timer
@logger
def calculate_large_sum(): 
    total = 0 
    for i in range(1000000): 
        total += i 
    
    return total

_ = calculate_large_sum()


print("\n" + "=" * 60)
print("Another function")
print("=" * 60)

@logger
@timer
def process_list(): 
    numbers = list(range(1000)) 
    return sorted(numbers, reverse=True)

_ = process_list()


ORDER 1: @logger then @timer
[START] Calling 'calculate_sum' with args=(), kwargs={}
Function calculate_sum took 0.006130 seconds to execute.
[END]   'calculate_sum' returned: 4999950000

ORDER 2: @timer then @logger
[START] Calling 'calculate_large_sum' with args=(), kwargs={}
[END]   'calculate_large_sum' returned: 499999500000
Function calculate_large_sum took 0.048623 seconds to execute.

Another function
[START] Calling 'process_list' with args=(), kwargs={}
Function process_list took 0.000045 seconds to execute.
[END]   'process_list' returned: [999, 998, 997, 996, 995, 994, 993, 992, 991, 990, 989, 988, 987, 986, 985, 984, 983, 982, 981, 980, 979, 978, 977, 976, 975, 974, 973, 972, 971, 970, 969, 968, 967, 966, 965, 964, 963, 962, 961, 960, 959, 958, 957, 956, 955, 954, 953, 952, 951, 950, 949, 948, 947, 946, 945, 944, 943, 942, 941, 940, 939, 938, 937, 936, 935, 934, 933, 932, 931, 930, 929, 928, 927, 926, 925, 924, 923, 922, 921, 920, 919, 918, 917, 916, 915, 914, 913, 912, 9

---

### Decorator Chaining – Why Order Matters

When you write two decorators like this:
```python
@logger
@timer
def calculate_sum():
    ...
```

Python applies them from bottom to top.

So the above code is the same as:
```python
calculate_sum = logger(timer(calculate_sum))
```


Important rule:
> The decorator closest to the function runs first (becomes the inner one).
> The decorator on top becomes the outer one.

---

Case 1: @logger then @timer
```python
@logger
@timer
def calculate_sum():
    ...
```

How Python builds it:
```python
calculate_sum = logger(timer(calculate_sum))
```

Structure:
```
logger (outer)
   └── timer (inner)
          └── original calculate_sum
```

What happens when you call calculate_sum():

- Logger starts → prints [START]
- Logger calls timer
- Timer starts → records start time
- Timer calls the original function
- Original function runs and returns the result
- Timer ends → prints the time taken
- Logger ends → prints [END]

Output looks like:
```
[START] Calling 'calculate_sum' ...
[TIMER] calculate_sum took 0.004x seconds
[END]   'calculate_sum' returned: 4999950000
```
---

Case 2: @timer then @logger
```python
@timer
@logger
def calculate_large_sum():
    ...
```

How Python builds it:
```python
calculate_large_sum = timer(logger(calculate_large_sum))
```

Structure:
```
timer (outer)
   └── logger (inner)
          └── original calculate_large_sum
```
What happens when you call calculate_large_sum():

- Timer starts → records start time
- Timer calls logger
- Logger starts → prints [START]
- Logger calls the original function
- Original function runs and returns the result
- Logger ends → prints [END]
- Timer ends → prints the time taken

Output looks like:
```
[START] Calling 'calculate_large_sum' ...
[END]   'calculate_large_sum' returned: 499999500000
[TIMER] calculate_large_sum took 0.042x seconds
```

| Order | Outer Decorator | Inner Decorator | What gets measured by timer|
| ------ | -------- | -------- | ------- |
|@logger then @timer | logger| timer | Only the original function |
|@timer then @logger | timer | logger | Function + logging print time |

---

# Q3: Generator-Based File Processing

Create:
```
read_file_lines(filename)
```
that yields one line at a time instead of loading the entire file into memory.

Then create:
```
filter_lines(lines, keyword)
```
that yields only lines containing a particular keyword.

---

**Input File**

Create: application.log

with:
```text
INFO User login successful
INFO Dashboard loaded
ERROR Database connection failed
INFO User updated profile
WARNING Disk space is getting low
ERROR Payment service unavailable
INFO User logged out
ERROR Timeout while connecting to database
INFO Report generated
WARNING API response is slow
```

---

**Test**

Search for:

keyword = "ERROR"

Then try:

keyword = "INFO"

Then:

keyword = "database"

Compare this approach with reading the entire file using:
```
readlines()
```

---

In [22]:
# ============================================================
# Generator-Based File Processing
# ============================================================

def read_file_lines(filename):
    with open(filename, "r") as file:
        for line in file:
            yield line.strip()


def filter_lines(lines, keyword):
    keyword = keyword.lower()
    for line in lines:
        if keyword in line.lower():
            yield line


if __name__ == "__main__":
    filename = "application.log"

    # ---------- Test 1: ERROR ----------
    print("=" * 50)
    print("Searching for: ERROR")
    print("=" * 50)
    lines = read_file_lines(filename)
    error_lines = filter_lines(lines, "ERROR")
    for line in error_lines:
        print(line)

    # ---------- Test 2: INFO ----------
    print("\n" + "=" * 50)
    print("Searching for: INFO")
    print("=" * 50)
    lines = read_file_lines(filename)
    info_lines = filter_lines(lines, "INFO")
    for line in info_lines:
        print(line)

    # ---------- Test 3: database ----------
    print("\n" + "=" * 50)
    print("Searching for: database")
    print("=" * 50)
    lines = read_file_lines(filename)
    db_lines = filter_lines(lines, "database")
    for line in db_lines:
        print(line)


    # ============================================================
    # Comparison with readlines()
    # ============================================================
    print("\n" + "=" * 50)
    print("Comparison with readlines()")
    print("=" * 50)

    # Method 1: readlines() - loads everything into memory
    with open(filename, "r") as file:
        all_lines = file.readlines()          # entire file loaded at once
        print(f"readlines() loaded {len(all_lines)} lines into memory")

    # Method 2: Generator - one line at a time
    print("Generator approach loads only one line at a time (memory efficient)")

Searching for: ERROR
ERROR Database connection failed
ERROR Payment service unavailable
ERROR Timeout while connecting to database

Searching for: INFO
INFO User login successful
INFO Dashboard loaded
INFO User updated profile
INFO User logged out
INFO Report generated

Searching for: database
ERROR Database connection failed
ERROR Timeout while connecting to database

Comparison with readlines()
readlines() loaded 10 lines into memory
Generator approach loads only one line at a time (memory efficient)


---

### Explanation

#### 1. `read_file_lines(filename)`
```python
def read_file_lines(filename):
    with open(filename, "r") as file:
        for line in file:
            yield line.strip()
```

- Uses `yield` → makes it a **generator**
- Reads **one line at a time**
- Does not store the whole file in memory
- `strip()` removes the newline character at the end of each line

#### 2. `filter_lines(lines, keyword)`
```python
def filter_lines(lines, keyword):
    for line in lines:
        if keyword.lower() in line.lower():
            yield line
```

- Takes a generator of lines as input
- Yields only the lines that contain the keyword
- Also a generator → still memory efficient
- Case-insensitive matching

#### 3. How they work together

```python
lines = read_file_lines("application.log")   # generator
error_lines = filter_lines(lines, "ERROR")   # another generator
for line in error_lines:                     # actual processing happens here
    print(line)
```

---

### What a for loop really does with a generator
When you write:
```python
for line in error_lines:
    print(line)
```

Python roughly does this behind the scenes:
```python
iterator = iter(error_lines)      # get iterator (generator is already an iterator)

while True:
    try:
        line = next(iterator)     # pull the next value
        print(line)
    except StopIteration:
        break                     # no more values → stop the loop
```
> So a 'for' loop is basically repeated 'next()' calls.

---

### How it works with generator chain
You have:
```python
lines = read_file_lines(filename)          # generator 1
error_lines = filter_lines(lines, "ERROR") # generator 2
```

When the for loop does next(error_lines):
- filter_lines wakes up and asks for a line from lines
- That means next(lines) is called internally
- read_file_lines reads one line from the file and yields it
- filter_lines receives that line
    - If it contains "ERROR" → yield it to the for loop
    - If not → ask for the next line again
- The for loop receives the yielded matching line and prints it
- Then the for loop calls next(error_lines) again… and the cycle repeats

---

### Flow of one next() call
```text
for loop
   │
   │  next(error_lines)
   ▼
filter_lines
   │
   │  next(lines)
   ▼
read_file_lines
   │
   │  reads one line from file
   ▼
yields line back to filter_lines
   │
   ▼
filter_lines checks the line
   │
   ├── if match → yield line to for loop
   └── if no match → ask for next line again
```

Important point
The generators do nothing until the for loop starts pulling values with next().
This is called lazy evaluation:

- No work happens at this moment:
```python
lines = read_file_lines(filename)
error_lines = filter_lines(lines, "ERROR")
```

- Work starts only when:
```python
for line in error_lines:    # ← pulling begins here
```

This is called **generator chaining** / **pipeline**:
- One generator feeds the next
- Data flows one item at a time

> The for loop keeps calling next() on the generator, and that single pull triggers the entire chain of generators to do their work one step at a time.

---

### Generator vs `readlines()`

| Feature                  | `readlines()`                     | Generator (`yield`)                |
|--------------------------|-----------------------------------|------------------------------------|
| Memory usage             | Loads entire file into memory     | Only one line in memory at a time  |
| Speed for large files    | Can be slow / crash on huge files | Efficient                          |
| Returns                  | A full list                       | A generator object                 |
| When processing starts   | After full file is read           | Immediately, line by line          |
| Best for                 | Small files                       | Large files / log processing       |

---

### Mental Model

```
File on disk
    ↓
read_file_lines()   → yields one line
    ↓
filter_lines()      → yields only matching lines
    ↓
for loop            → prints the final results
```

---

> **Note:** For a more detailed explanation, visit: [Generator and Decorator Folder in OOPs](../5_OOPs/8_Generator_Decorator)
